1. Import Libraries

In [2]:
## kernel - ids
"""
Persist Salesforce object field metadata (df_meta) into a SQL Server table.

- Creates the table if it doesn't exist
- Appends new rows OR performs an UPSERT (MERGE) if enabled
- Uses fast_executemany for performance
"""

#Importing Other Libraries
from simple_salesforce import Salesforce,SalesforceLogin,SalesforceAuthenticationFailed,SFType
import pandas as pd
import json
import numpy as np
import time as t
import uuid
import os
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from collections import OrderedDict

from core_functions import *
from sf_source_objects import *
from sf_target_objects import *

In [4]:
# ----------------------------
# PARAMETERS
# 1) CONFIGURATION
# ----------------------------
p_db_server  = 'Rhombus-SQL-Ser'
# p_db  = 'ADVICE_ETL_PRD'
p_db  = 'SalesforceMigration'
p_sch = 'ids'
p_tbl_fd = 'sf_object_fd'
p_tbl_pl = 'sf_object_pl'
# p_sf_src = 'hubuat'
p_sf_src = 'hubprod'
# p_sf_tgt = 'rhpartial'
# p_sf_tgt = 'rhdata'
# p_sf_tgt = 'rhuat'
# p_sf_tgt = 'datasb'
# p_sf_tgt = 'flow2sb'
p_sf_tgt = 'rhsit'

# ----------------------------
# CONSTANTS
# WRITE: APPEND or UPSERT
# ----------------------------
UPSERT_ENABLED = True  # set False to just append/replace
WRITE_CHUNK_SIZE = 10_000

connection_string = (
    f"mssql+pyodbc://@{p_db_server}/{p_db}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
    "&Trusted_Connection=yes"
    "&fast_executemany=True"
)


## Configure Logging
This section initializes a logging framework to monitor the ETL workflow’s execution and facilitate debugging by:
Creating an isolated logger named ETL_Salesforce.
Setting the log level to INFO to capture all important messages.
Outputting formatted messages (with timestamps and levels) to the console.
Disabling propagation to prevent duplicate log entrie

In [5]:
# Configuring logging
import logging
 
# Create a dedicated logger for the ETL process
logger = logging.getLogger('ETL_Salesforce')
logger.setLevel(logging.INFO)
 
# Clear any existing handlers to avoid duplicates
logger.handlers = []  # Remove all handlers from this logger
 
# Create a new StreamHandler with the desired format
handler = logging.StreamHandler()
handler.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s"))
logger.addHandler(handler)
 
# Disable propagation to prevent root logger interference
logger.propagate = False

## Establish Connections
This section establishes a connection to the source Salesforce environment using credentials securely fetched from Google Colab’s secrets storage.
Connecting to the Source Salesforce Instance:

In [4]:
#Connecting To The Source Salesforce Instance
from secrets_ids import secrets
url = secrets.get(p_sf_src, {}).get('site', 'Unknown')
username = secrets.get(p_sf_src, {}).get('username', 'Unknown')
pswd = secrets.get(p_sf_src, {}).get('password', 'Unknown')
token = secrets.get(p_sf_src, {}).get('token', 'Unknown')

src_sf_instance = connect_to_salesforce(url=url, username=username, password=pswd, token=token, domain='login', logger=logger)
 
logger.info(f"Source SF Instance Session ID = {src_sf_instance.session_id}")

2026-09-03 11:39:13,721 [INFO] Connected successfully to https://ioofadviserhub.my.salesforce.com
2026-09-03 11:39:13,722 [INFO] Source SF Instance Session ID = 00D6F0000029Bzd!AQEAQFvKAVv8xUIEq9pP4LuifIq4ybyy9LLCUzK094I_fMS87.vmiV.MtJ2rIWSurGeD1.x76u8DOoiIcEyzBOMd0rYOUIzs


To maintain security, credentials are not hardcoded. Instead, they are stored in Google Colab’s Secrets vault. The application dynamically accesses these stored values (src_username, src_password, and src_token) via the userdata.get('secret_name') method. The fetched credentials are passed to the connect_to_salesforce function to authenticate and create a Salesforce client instance.

Connecting to the Destination Salesforce Instance:

In [6]:
#Connecting To The Destination Salesforce Instance
from secrets_ids import secrets
url_tgt = secrets.get(p_sf_tgt, {}).get('site', 'Unknown')
username_tgt = secrets.get(p_sf_tgt, {}).get('username', 'Unknown')
pswd_tgt = secrets.get(p_sf_tgt, {}).get('password', 'Unknown')
token_tgt = secrets.get(p_sf_tgt, {}).get('token', 'Unknown')

dst_sf_instance = connect_to_salesforce(url=url_tgt, username=username_tgt, password=pswd_tgt, token=token_tgt, domain='test', logger=logger)
 
logger.info(f"Destination SF Instance Session ID = {dst_sf_instance.session_id}")

2026-09-03 21:52:30,188 [INFO] Connected successfully to https://test.salesforce.com
2026-09-03 21:52:30,189 [INFO] Destination SF Instance Session ID = 00DBm00000CWPlx!AQEAQJisfayl5n7.FwzSKP.ctprm4b84oZySLRyBaIa8MAzE8jvHcOxYPPc05qLr3sPwjfiGuz76lF5JCcIqb2.XZT8KKTbS


## Data Extraction Phase
This section handles the data extraction process by:
Defining the Salesforce object to query (Account) and the key reference column (Name) used for data validation and uniqueness checks.
Constructing a SOQL query string to retrieve the Id, Name, and Phone fields while limiting the results to three records to get a small sample of data for testing.
Calling the extract_query function with the Salesforce connection and SOQL query; This function executes the query and loads the results into a panda DataFrame (object_df).
Logging the count of non-null values per column to identify missing data.
Logging the total number of records extracted.
Logging the count of unique values in the reference column (Name).
Checking for duplicates in the reference column and logging a warning if any are found.
Displaying the first five rows of the extracted DataFrame to give a preview of the data structure and content.

In [11]:
from sf_target_objects import TARGET_OBJECTS

lst_objects_copy = TARGET_OBJECTS

In [21]:
lst_objects_copy = [
'Profile'
# ,'ProfileSkill'
# ,'ProfileSkillEndorsement'
# ,'ProfileSkillUser'
# ,'Prompt'
# ,'PromptAction'
# ,'PromptError'
# ,'PromptVersion'
,'PSA__c'
# ,'Publisher'
# ,'PushTopic'
,'QA__c'
,'QA_Assignment_Hisotry__c'
# ,'QDC_Training__c'
,'Qualification__c'
,'Qualification_Type__c'
,'Question_Group__c'
,'Question_Responses__c'
# ,'QueueSobject'
# ,'QuickText'
# ,'QuickTextUsage'
# ,'Quote'
# ,'QuoteDocument'
# ,'QuoteLineItem'
# ,'QuoteTemplateRichTextData'
# ,'RAM_Newsletter__c'
# ,'RAM_Notes__c'
# ,'RAM_Portfolio_Performance__mdt'
# ,'RecAlrtDataSrcExpSetDef'
# ,'RecentlyViewed'
# ,'Recommendation'
# ,'RecommendationResponse'
# ,'RecordAction'
# ,'RecordActnSelItemExtrc'
# ,'RecordAlert'
# ,'RecordAlertActionableTarget'
# ,'RecordAlertCategory'
# ,'RecordAlertDataSource'
# ,'RecordAlertTemplate'
,'RecordType'
# ,'RecordTypeIds__c'
# ,'RedirectWhitelistUrl'
# ,'Referral_Director__c'
# ,'Referral_Partner__c'
,'Refund_Request__c'
,'Registered_Training_Organisation__c'
# ,'RelatedListColumnDefinition'
# ,'RelatedListDefinition'
# ,'RelationshipDomain'
# ,'RelationshipInfo'
# ,'ReplyEmailSettings'
# ,'Report'
# ,'ReportAnomalyEvent'
# ,'ReportAnomalyEventStore'
# ,'ReportEvent'
# ,'ReportEventStream'
# ,'ReputationLevel'
# ,'ReputationPointsRule'
,'Response_Finding__c'
# ,'Reviewer_Accreditation__c'
,'Reviews_Accounted__c'
# ,'Root_Cause_Indicator_Value__mdt'
# ,'Root_Cause_Multiplier__c'
# ,'SalesforceContract'
# ,'SalesforceInvoice'
# ,'SalesforcePayment'
# ,'SalesforceQuote'
# ,'SamlSsoConfig'
# ,'Scontrol'
# ,'Scorecard'
# ,'ScorecardAssociation'
# ,'ScorecardMetric'
# ,'SearchActivity'
# ,'SearchLayout'
# ,'SearchPromotionRule'
,'SecondaryPracticeAdviserHistory__c'
# ,'SecurityCustomBaseline'
,'SelectedQuestions_Targetedreviews__c'
# ,'ServiceContract'
# ,'ServiceSetupProvisioning'
# ,'SessionHijackingEvent'
# ,'SessionHijackingEventStore'
# ,'SessionPermSetActivation'
# ,'SetupAssistantStep'
# ,'SetupAuditTrail'
# ,'SetupDataSynchronization'
# ,'SetupEntityAccess'
# ,'Site'
# ,'SiteDetail'
# ,'SiteIframeWhiteListUrl'
# ,'SiteMarketingDataExtensionMapping'
# ,'SiteRedirectMapping'
# ,'SlackChannelRelatedRecord'
# ,'SlaProcess'
,'Snapshot_Data__c'
# ,'SocialPersona'
# ,'SocialPost'
# ,'Solution'
# ,'SolutionStatus'
# ,'sortablegrid__SDG__c'
# ,'sortablegrid__SDG_Action__c'
# ,'sortablegrid__SDG_Field__c'
# ,'sortablegrid__sdg_Preferences__c'
# ,'sortablegrid__SDGDefaults__mdt'
,'Special_Conditions__c'
# ,'SPSamlAttributes'
# ,'Stamp'
# ,'StampAssignment'
# ,'StaticResource'
# ,'StreamingChannel'
# ,'TabDefinition'
# ,'TableauHostMapping'
,'Target__c'
,'Task'
# ,'TaskPriority'
,'TaskRelation'
# ,'TaskStatus'
# ,'TaskWhoRelation'
# ,'TenantUsageEntitlement'
# ,'TestSuiteMembership'
,'Third_Party_Provider__c'
# ,'ThirdPartyAccountLink'
# ,'ThreatDetectionFeedback'
# ,'TimelineObjectDefinition'
# ,'TodayGoal'
# ,'Toggle_CSA_Switch_Off_Creation__mdt'
# ,'Topic'
,'Topic__c'
# ,'TopicAssignment'
# ,'TopicUserEvent'
,'Tracking__c'
# ,'TransactionSecurityPolicy'
# ,'Translation'
# ,'Trigger_Flags__c'
# ,'TwoFactorAuthenticationSettings__mdt'
# ,'UiFormulaCriterion'
# ,'UiFormulaRule'
# ,'UndecidedEventRelation'
# ,'Universal_Audit_Log__c'
# ,'Universal_User_Lookup__mdt'
# ,'unleashformula__Formula_Mapping__c'
# ,'UriEvent'
# ,'UriEventStream'
,'User'
# ,'UserAppInfo'
# ,'UserAppMenuCustomization'
# ,'UserAppMenuItem'
# ,'UserCustomBadge'
# ,'UserDefinedLabel'
# ,'UserDefinedLabelAssignment'
# ,'UserEmailPreferredPerson'
# ,'UserEntityAccess'
# ,'UserFieldAccess'
,'UserLicense'
# ,'UserListView'
# ,'UserListViewCriterion'
# ,'UserLocalWebServerIdentity'
# ,'UserLogin'
# ,'UserPackageLicense'
# ,'UserPermissionAccess'
# ,'UserPreference'
# ,'UserProvAccount'
# ,'UserProvAccountStaging'
# ,'UserProvisioningConfig'
# ,'UserProvisioningLog'
# ,'UserProvisioningRequest'
# ,'UserProvMockTarget'
# ,'UserRecordAccess'
,'UserRole'
# ,'UserSetupEntityAccess'
# ,'UserSharedFeature'
# ,'VerificationHistory'
,'Vetting_Result__c'
# ,'VisualforceAccessMetrics'
# ,'Vote'
# ,'Watchlist_Information__c'
# ,'WaveAssetEvent'
# ,'WaveAutoInstallRequest'
# ,'WaveCompatibilityCheckItem'
# ,'Wealth_Advice_Referral_Program__c'
# ,'Wealth_Central__c'
,'Wealth_Central_Adoption__c'
# ,'WebLink'
# ,'WorkAccess'
# ,'WorkBadge'
# ,'WorkBadgeDefinition'
,'Workshop__c'
]


In [39]:
lst_objects_copy = [
# 'Account'
# 'Contact'
'CSA_Fee_Switch_Off_History__c'
# 'Task'1
# 'Consequence_Management_Forum_Entry__c'
# 'UserRole'
# 'RecordType'
# 'CSA_Case__c'
# 'CSA_Fee_Switch_Off_History__c'
# 'PermissionSetAssignment'
# 'PermissionSet'
# 'PermissionSetGroup'
# ,'PermissionSetLicense'
# ,'UserLicense'
]
filter_by_license = False

for sfo in lst_objects_copy:
    print(f"Processing {sfo}.")
    q_sql_select = f"""
    WITH cols AS (
        SELECT
            [object_api_name],
            [column_list] = STRING_AGG(convert(varchar(max),[field_api_name]),', ')
                WITHIN GROUP (ORDER BY [object_api_name])
        FROM [SalesforceMigration].[ids].[sf_object_meta_ra] (nolock)
        where 1=1
        and [object_api_name] = '{sfo}'
        GROUP BY [object_api_name]
    )
    SELECT
        'SELECT '
            + [column_list] + CHAR(13) + CHAR(10) 
            + ' FROM ' + [object_api_name] 
    FROM cols
    ORDER BY [object_api_name]
    """

    # Merge into table
    engine = get_engine(connection_string)
    with engine.begin() as conn:
        rst_select = conn.execute(text(q_sql_select))
        # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
        # columns = rst_select.keys()
        rows = rst_select.fetchall()
        soql_select = rows[0][0]
        # soql_select = soql_select + (" WHERE Licensee__c in ('Bridges','Shadforth','Navara')" if filter_by_license else "")
        # soql_select = soql_select + (" AND Is_Active__c = 'Yes' ")
        # soql_select = soql_select + (" AND Commencement_Date__c >= 2026-01-01 and Commencement_Date__c < 2028-01-01 ")
        # soql_select = soql_select + (" AND Commencement_Date__c != NULL ")
        # soql_select = soql_select + (" AND Adviser_Name__c >= 'Z' ")
        print(soql_select)
        # ls_target_tbl = df_target_tbl.columns.tolist()
        conn.commit()
        # conn.close()

    df_object = extract_query(dst_sf_instance,soql_select,logger=logger)

    if df_object.empty:
        print(f"No data found for {sfo}.")
    else:
        df_object_normalized = normalize_df(df_object)
        engine = get_engine(connection_string)
        with engine.begin() as conn:
            # write_upsert(engine, df_final, q_sql_createTable_fd, temp_fd, q_sql_createTemp_fd, q_sql_merge_fd)
            df_object_normalized.to_sql(
                schema='sfrhd',
                name=sfo,
                con=engine,  # use raw connection for temp table visibility
                if_exists="append",
                index=False
            )


Processing CSA_Fee_Switch_Off_History__c.
2026-09-04 02:11:34,568 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-04 02:11:34,568 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-04 02:11:34,568 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-04 02:11:34,568 INFO sqlalchemy.engine.Engine [generated in 0.00094s] ()
2026-09-04 02:11:34,568 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-04 02:11:34,568 INFO sqlalchemy.engine.Engine [generated in 0.00098s] ()
2026-09-04 02:11:34,583 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-04 02:11:34,583 INFO sqlalchemy.engine.Engine [generated in 0.00148s] ()
2026-09-04 02:11:34,587 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-04 02:11:34,589 INFO sqlalchemy.engine.Engine 
    WITH cols AS (
        SELECT
            [object_api_name],
            [

2026-09-04 02:11:34,590 [INFO] Executing SOQL query: SELECT Adviser__c, ASF_CSA_Case__c, ASF_CSA_Case_dv__c, CreatedById, CreatedDate, CSA_Case__c, CSA_Case_dv__c, Email_Log__c, Email_Sent_Status__c, Email_sub_status__c, Error_Message__c, Id, IsDeleted, LastActivityDate, LastModifiedById, LastModifiedDate, LastReferencedDate, LastViewedDate, Licensee__c, Name, NameLicenseeKey__c, OwnerId, Policy_Number__c, Processed_to_send_email__c, Supplier_Name__c, SystemModstamp, Thread_ID__c, Unique_Key__c, Wave_Internal__c
 FROM CSA_Fee_Switch_Off_History__c
2026-09-04 02:11:35,313 [INFO] Fetching next batch: /services/data/v59.0/query/0r8xx5qZnudF55zAEC-2000
2026-09-04 02:11:35,620 [INFO] Query complete. Rows extracted: 3067 . Time taken: 1.03s


2026-09-04 02:11:35,683 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-04 02:11:35,684 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-04 02:11:35,686 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-04 02:11:35,687 INFO sqlalchemy.engine.Engine [generated in 0.00065s] ()
2026-09-04 02:11:35,689 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-04 02:11:35,690 INFO sqlalchemy.engine.Engine [generated in 0.00098s] ()
2026-09-04 02:11:35,692 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-04 02:11:35,692 INFO sqlalchemy.engine.Engine [generated in 0.00068s] ()
2026-09-04 02:11:35,694 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-04 02:11:35,702 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-04 02:11:35,717 INFO sqlalchemy.engine.Engine SELECT [INFORMATION_SCHEMA].[TABLES].[TABLE_NAME] 